# ESP8266 Signal Lab — scan analysis

Analyze only captures collected with authorization. This notebook stays local; do not publish SSIDs, BSSIDs, or precise GPS locations without permission. Presence/RSSI changes are heuristics, not proof of attacks.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Update this path to your local capture file.
CSV_PATH = Path('../data/scan.csv')
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
# Normalize types and discard rows missing the essential AP identifier or signal level.
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['rssi_dbm'] = pd.to_numeric(df['rssi_dbm'], errors='coerce')
df['channel'] = pd.to_numeric(df['channel'], errors='coerce')
df = df.dropna(subset=['timestamp', 'bssid', 'rssi_dbm']).copy()
print(f'{len(df)} observations from {df.bssid.nunique()} distinct BSSIDs')

In [ ]:
# Channel distribution and RSSI summary.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['channel'].dropna().astype(int).value_counts().sort_index().plot.bar(ax=axes[0], title='Observations by channel')
df['rssi_dbm'].plot.hist(bins=20, ax=axes[1], title='Signal strength (dBm)')
axes[0].set_xlabel('Channel'); axes[1].set_xlabel('RSSI (dBm)')
plt.tight_layout()

In [ ]:
# Review unusually large per-BSSID signal changes. Large deltas can have benign causes.
ordered = df.sort_values('timestamp')
ordered['rssi_delta_db'] = ordered.groupby('bssid')['rssi_dbm'].diff().abs()
review = ordered[ordered['rssi_delta_db'] >= 20][['timestamp', 'ssid', 'bssid', 'rssi_dbm', 'rssi_delta_db', 'channel']]
review.sort_values('rssi_delta_db', ascending=False).head(30)

In [ ]:
# BSSID presence changes across collection batches. This is a survey heuristic only.
# The collector timestamp is per observation; round to 30-second windows to group scan results.
ordered['window'] = ordered['timestamp'].dt.floor('30s')
presence = ordered.drop_duplicates(['window', 'bssid']).pivot_table(index='window', columns='bssid', values='rssi_dbm', aggfunc='mean')
presence = presence.sort_index()
changes = presence.notna().astype(int).diff().fillna(0)
events = changes.stack().rename('presence_change').reset_index()
events = events[events['presence_change'] != 0].sort_values('window')
events.head(30)

## Optional location plot

Only plot location data when collection was explicitly authorized. Keep the plot and source CSV private; precise coordinates can reveal sensitive locations.

In [ ]:
if {'latitude', 'longitude'}.issubset(df.columns):
    geo = df.copy()
    geo['latitude'] = pd.to_numeric(geo['latitude'], errors='coerce')
    geo['longitude'] = pd.to_numeric(geo['longitude'], errors='coerce')
    geo = geo.dropna(subset=['latitude', 'longitude'])
    if not geo.empty:
        ax = geo.plot.scatter(x='longitude', y='latitude', c='rssi_dbm', colormap='viridis',
                              figsize=(7, 6), colorbar=True, title='Authorized survey — location-sensitive')
        ax.set_aspect('equal', adjustable='datalim')
        plt.show()
    else:
        print('No valid GPS fixes in this capture.')
else:
    print('No GPS columns available.')